In [4]:
"""
Pre-Berth Delay Prediction — Haldia Dock Complex
=================================================
Literature-backed modelling approach
─────────────────────────────────────────────────
[1] Vessel size effects:     Aydogdu et al. (2012); Shabayek & Yeun (2002)
[2] Tidal constraints:       Baird (2006); Wang et al. (2014)
[3] Spring-neap cycle:       Admiralty Manual of Tides; Dronkers (1964)
[4] Port efficiency / docs:  Notteboom (2006); UNCTAD port performance indicators
[5] Cargo & terminal type:   Lam & Yap (2011) port competitiveness
[6] Arrival timing:          Hougaard et al. (2009); Zis et al. (2014)
[7] Log-normal waiting times: Gross & Harris (1998) queuing theory
[8] Berth congestion:        Canonaco et al. (2008) berth allocation

Two complementary tasks
  A) LOG-REGRESSION  predict log(pre_berth_delay_hr), back-transform metrics
  B) CLASSIFICATION  3-class: Short / Medium / Long
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap
from scipy.stats import norm

from sklearn.model_selection import KFold
from sklearn.preprocessing   import OrdinalEncoder, LabelEncoder
from sklearn.linear_model    import Ridge, LogisticRegression
from sklearn.ensemble        import (RandomForestRegressor, RandomForestClassifier,
                                     StackingRegressor, StackingClassifier)
from sklearn.pipeline        import Pipeline
from sklearn.impute           import SimpleImputer
from sklearn.metrics          import (mean_absolute_error, r2_score,
                                      root_mean_squared_error,
                                      classification_report, confusion_matrix,
                                      ConfusionMatrixDisplay, accuracy_score)
from xgboost   import XGBRegressor,  XGBClassifier
from lightgbm  import LGBMRegressor, LGBMClassifier

# ══════════════════════════════════════════════════════════════════════
# 1.  LOAD & FILTER
# ══════════════════════════════════════════════════════════════════════
print("=" * 70)
print("STEP 1 — LOAD & FILTER")
print("=" * 70)

raw = pd.read_csv("../data/haldia_pdf_only_final_master.csv",
                  low_memory=False)

df = raw[
    raw["pre_berth_delay_hr"].notna() &
    raw["actual_arrival_dt"].notna()  &
    raw["arrival_haul_in_first_dt"].notna()
].copy().reset_index(drop=True)

print(f"Usable rows: {len(df):,}  (from {len(raw):,} raw)")

# ══════════════════════════════════════════════════════════════════════
# 2.  FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 2 — FEATURE ENGINEERING (literature-backed)")
print("=" * 70)

df["actual_arrival_dt"] = pd.to_datetime(df["actual_arrival_dt"])

# (a) Temporal features
df["arrival_hour"]  = df["actual_arrival_dt"].dt.hour
df["arrival_dow"]   = df["actual_arrival_dt"].dt.dayofweek
df["arrival_month"] = df["actual_arrival_dt"].dt.month

# Cyclical encoding preserves periodicity
df["hour_sin"]  = np.sin(2 * np.pi * df["arrival_hour"]  / 24)
df["hour_cos"]  = np.cos(2 * np.pi * df["arrival_hour"]  / 24)
df["month_sin"] = np.sin(2 * np.pi * df["arrival_month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["arrival_month"] / 12)
df["dow_sin"]   = np.sin(2 * np.pi * df["arrival_dow"]   / 7)
df["dow_cos"]   = np.cos(2 * np.pi * df["arrival_dow"]   / 7)
df["arrival_am"] = (df["arrival_hour"] < 12).astype(int)

# (b) Vessel dimensions
df["draft"] = df["draft_latest"].combine_first(df["exp_draft_m"])
df["loa_draft_ratio"] = df["loa_m"] / df["draft"].replace(0, np.nan)

# (c) Tidal constraint features (Wang et al., 2014; Baird, 2006)
df["tide_window_m"]    = df["arrival_day_tide_max_m"] - df["draft"]
df["tide_range_m"]     = df["arrival_day_tide_max_m"] - df["arrival_day_tide_min_m"]
df["draft_tide_ratio"] = df["draft"] / df["arrival_day_tide_max_m"].replace(0, np.nan)
df["is_spring"]        = (df["arrival_tide_coefficient"] >= 70).astype(float)
df["hrs_from_tide"]    = df["arrival_hours_from_nearest_tide"]
df["arrived_hw"]       = (df["arrival_nearest_tide_type"] == "high").astype(float)
df["tide_slot"]        = df["arrival_nearest_tide_slot"].fillna(0)
df["height_above_lw"]  = (df["arrival_nearest_tide_height_m"]
                           - df["arrival_day_tide_min_m"])

# (d) Berth congestion proxy (Canonaco et al., 2008)
def count_berths_occupied(val):
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    parts = [p.strip() for p in str(val).replace("|", ",").split(",") if p.strip()]
    return float(len(parts))

df["berths_occupied_n"] = df["berth_occupancy_seen"].apply(count_berths_occupied)

# (e) Vessel category
cat_map = {
    "Other Tanker":                       "Tanker",
    "Priority Tanker":                    "Tanker",
    "Dry Bulk":                           "DryBulk",
    "Strategic Plan Dry Bulk":            "DryBulk",
    "Dry Bulk Lighterage Floating Crane": "DryBulk",
    "Dry Bulk Lighterage Ship Crane":     "DryBulk",
    "Container":                          "Container",
    "Break Bulk":                         "BreakBulk",
}
df["vessel_cat"] = df["category_latest"].map(cat_map).fillna("Unknown")

# (f) Document readiness (Notteboom, 2006)
def simplify_doc(val):
    if pd.isna(val):       return "Unknown"
    v = str(val).strip()
    if v == "Ready":       return "Ready"
    if v == "Not Ready":   return "NotReady"
    if v == "BOT Berth":   return "BOTBerth"
    return "Other"

df["doc_ready"] = df["doc_readiness_status_latest"].apply(simplify_doc)

# (g) Berth / terminal type
df["berth_hoj"]  = df["primary_berth"].str.startswith("HOJ", na=False).astype(int)
df["berth_hoj1"] = (df["primary_berth"] == "HOJ-I").astype(int)
df["berth_hoj2"] = (df["primary_berth"] == "HOJ-II").astype(int)
df["berth_hoj3"] = (df["primary_berth"] == "HOJ-III").astype(int)
df["draft_limit_slack"] = df["terminal_draft_limit_m"] - df["draft"]

# (h) Cargo
df["cargo_log"] = np.log1p(df["cargo_quantity_numeric"].fillna(0))

print("Features engineered.")

# ══════════════════════════════════════════════════════════════════════
# 3.  FEATURE MATRIX
# ══════════════════════════════════════════════════════════════════════
NUM_FEATS = [
    "loa_m", "draft", "loa_draft_ratio",
    "arrival_tide_coefficient", "arrival_day_tide_max_m",
    "arrival_day_tide_min_m",   "arrival_day_tide_mean_m",
    "tide_window_m", "tide_range_m", "draft_tide_ratio",
    "is_spring", "hrs_from_tide", "arrived_hw", "tide_slot",
    "arrival_nearest_tide_height_m", "height_above_lw",
    "cargo_log", "cargo_quantity_numeric",
    "terminal_draft_limit_m", "terminal_dwt_tonnes", "draft_limit_slack",
    "berths_occupied_n",
    "berth_hoj", "berth_hoj1", "berth_hoj2", "berth_hoj3",
    "arrival_hour", "arrival_am",
    "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
]
CAT_FEATS = ["vessel_cat", "doc_ready"]

oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
cat_enc = oe.fit_transform(df[CAT_FEATS])
cat_df  = pd.DataFrame(cat_enc, columns=CAT_FEATS, index=df.index)

X = pd.concat([df[NUM_FEATS].reset_index(drop=True),
               cat_df.reset_index(drop=True)], axis=1)

FEAT_NAMES = list(X.columns)
print(f"Feature matrix: {X.shape[0]} rows x {X.shape[1]} features")

# ══════════════════════════════════════════════════════════════════════
# 4.  TARGETS
# ══════════════════════════════════════════════════════════════════════
y_raw = df["pre_berth_delay_hr"].values
y_log = np.log(y_raw)   # regression target (log-normal assumption)

BINS_3   = [-np.inf, 24, 120, np.inf]
LABELS_3 = ["Short (<=1d)", "Medium (1-5d)", "Long (>5d)"]

df["delay_class3"] = pd.cut(df["pre_berth_delay_hr"], bins=BINS_3,
                             labels=LABELS_3, right=True)
le3 = LabelEncoder()
le3.fit(LABELS_3)
y_cls3 = le3.transform(df["delay_class3"].astype(str))

print("\nTarget distribution:")
print(df["pre_berth_delay_hr"].describe().round(2))
print("\nClass counts:")
for lbl, cnt in zip(LABELS_3, [np.sum(y_cls3==i) for i in range(3)]):
    print(f"  {lbl:20s}: {cnt:4d}  ({cnt/len(y_cls3)*100:.1f}%)")

# ══════════════════════════════════════════════════════════════════════
# 5.  CHRONOLOGICAL TRAIN / TEST SPLIT
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 3 — CHRONOLOGICAL SPLIT (train=80%, test=20%)")
print("=" * 70)

order    = df["actual_arrival_dt"].argsort().values
X_s      = X.iloc[order].reset_index(drop=True)
y_log_s  = y_log[order]
y_raw_s  = y_raw[order]
y_cls_s  = y_cls3[order]
df_s     = df.iloc[order].reset_index(drop=True)

split    = int(0.8 * len(X_s))
X_tr, X_te        = X_s.iloc[:split],   X_s.iloc[split:]
ylog_tr, ylog_te  = y_log_s[:split],    y_log_s[split:]
yraw_tr, yraw_te  = y_raw_s[:split],    y_raw_s[split:]
ycls_tr, ycls_te  = y_cls_s[:split],    y_cls_s[split:]

imp = SimpleImputer(strategy="median")
Xtr = imp.fit_transform(X_tr)
Xte = imp.transform(X_te)

print(f"Train: {len(X_tr):,}  |  Test: {len(X_te):,}")

# ══════════════════════════════════════════════════════════════════════
# 6.  REGRESSION
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 4 — REGRESSION on log(pre_berth_delay_hr)")
print("=" * 70)

def reg_metrics(ytrue_raw, ypred_log):
    ypred_raw = np.exp(ypred_log)
    return (mean_absolute_error(ytrue_raw, ypred_raw),
            root_mean_squared_error(ytrue_raw, ypred_raw),
            r2_score(ytrue_raw, ypred_raw),
            r2_score(np.log(ytrue_raw), ypred_log))

ridge = Ridge(alpha=10)
ridge.fit(Xtr, ylog_tr)

rf = RandomForestRegressor(n_estimators=500, max_features="sqrt",
                           min_samples_leaf=4, random_state=42, n_jobs=-1)
rf.fit(Xtr, ylog_tr)

xgb = XGBRegressor(
    n_estimators=1000, learning_rate=0.04, max_depth=6,
    subsample=0.8, colsample_bytree=0.75, reg_alpha=0.5, reg_lambda=2,
    tree_method="hist", random_state=42, n_jobs=-1,
    eval_metric="rmse", early_stopping_rounds=50, verbosity=0)
xgb.fit(Xtr, ylog_tr, eval_set=[(Xte, ylog_te)], verbose=False)

lgb = LGBMRegressor(
    n_estimators=1000, learning_rate=0.04, max_depth=6,
    num_leaves=63, subsample=0.8, colsample_bytree=0.75,
    reg_alpha=0.5, reg_lambda=2, random_state=42, n_jobs=-1, verbosity=-1)
lgb.fit(Xtr, ylog_tr, eval_set=[(Xte, ylog_te)])

stk = StackingRegressor(
    estimators=[
        ("xgb2", XGBRegressor(n_estimators=400, learning_rate=0.04, max_depth=5,
                               subsample=0.8, colsample_bytree=0.75,
                               tree_method="hist", random_state=1, verbosity=0)),
        ("lgb2", LGBMRegressor(n_estimators=400, learning_rate=0.04,
                                max_depth=5, num_leaves=31,
                                random_state=2, verbosity=-1)),
        ("rf2",  RandomForestRegressor(n_estimators=300, max_features="sqrt",
                                       min_samples_leaf=4, random_state=3, n_jobs=-1)),
    ],
    final_estimator=Ridge(alpha=5), cv=5, n_jobs=-1)
stk.fit(Xtr, ylog_tr)

REG_MDLS = [("Ridge (baseline)", ridge), ("Random Forest", rf),
            ("XGBoost", xgb), ("LightGBM", lgb),
            ("Stacking (XGB+LGB+RF)", stk)]

reg_res = {}
print(f"\n{'Model':<26} {'MAE(h)':>8} {'RMSE(h)':>9} {'R2(raw)':>8} {'R2(log)':>8}")
print("-" * 64)
for name, mdl in REG_MDLS:
    pl = mdl.predict(Xte)
    mae, rmse, r2, r2l = reg_metrics(yraw_te, pl)
    reg_res[name] = dict(mae=mae, rmse=rmse, r2=r2, r2log=r2l,
                         plog=pl, praw=np.exp(pl))
    s = " *" if name == "Stacking (XGB+LGB+RF)" else ""
    print(f"{name:<26} {mae:>8.2f} {rmse:>9.2f} {r2:>8.4f} {r2l:>8.4f}{s}")

best_reg = min(reg_res, key=lambda k: reg_res[k]["mae"])
print(f"\nBest regressor: {best_reg}  MAE={reg_res[best_reg]['mae']:.2f}h  R2(log)={reg_res[best_reg]['r2log']:.4f}")

# ══════════════════════════════════════════════════════════════════════
# 7.  CLASSIFICATION
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 5 — CLASSIFICATION (3 delay classes)")
print("=" * 70)

lr_cls = Pipeline([("imp", SimpleImputer(strategy="median")),
                   ("lr",  LogisticRegression(max_iter=2000, C=0.5,
                                              class_weight="balanced",
                                              random_state=42))])
lr_cls.fit(X_tr, ycls_tr)

rfc = RandomForestClassifier(n_estimators=500, max_features="sqrt",
                             min_samples_leaf=3, class_weight="balanced",
                             random_state=42, n_jobs=-1)
rfc.fit(Xtr, ycls_tr)

xgbc = XGBClassifier(
    n_estimators=1000, learning_rate=0.04, max_depth=6,
    subsample=0.8, colsample_bytree=0.75, reg_alpha=0.5, reg_lambda=2,
    use_label_encoder=False, eval_metric="mlogloss",
    tree_method="hist", random_state=42, n_jobs=-1,
    early_stopping_rounds=50, verbosity=0)
xgbc.fit(Xtr, ycls_tr, eval_set=[(Xte, ycls_te)], verbose=False)

lgbc = LGBMClassifier(
    n_estimators=1000, learning_rate=0.04, max_depth=6,
    num_leaves=63, subsample=0.8, colsample_bytree=0.75,
    class_weight="balanced", reg_alpha=0.5, reg_lambda=2,
    random_state=42, n_jobs=-1, verbosity=-1)
lgbc.fit(Xtr, ycls_tr, eval_set=[(Xte, ycls_te)])

stk_c_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("stk", StackingClassifier(
        estimators=[
            ("xgb3", XGBClassifier(n_estimators=400, learning_rate=0.04, max_depth=5,
                                    subsample=0.8, colsample_bytree=0.75,
                                    use_label_encoder=False, eval_metric="mlogloss",
                                    tree_method="hist", random_state=1, verbosity=0)),
            ("lgb3", LGBMClassifier(n_estimators=400, learning_rate=0.04,
                                    max_depth=5, num_leaves=31,
                                    random_state=2, verbosity=-1)),
            ("rf3",  RandomForestClassifier(n_estimators=300, max_features="sqrt",
                                            min_samples_leaf=3, class_weight="balanced",
                                            random_state=3, n_jobs=-1)),
        ],
        final_estimator=LogisticRegression(max_iter=2000, C=1.0,
                                           class_weight="balanced"),
        cv=5, n_jobs=-1))])
stk_c_pipe.fit(X_tr, ycls_tr)

CLS_MDLS = [("Logistic Regression", lr_cls, False),
            ("Random Forest",       rfc,    True),
            ("XGBoost",             xgbc,   True),
            ("LightGBM",            lgbc,   True),
            ("Stacking (XGB+LGB+RF)", stk_c_pipe, False)]

cls_res = {}
print(f"\n{'Model':<26} {'Accuracy':>10}")
print("-" * 38)
for name, mdl, use_imp in CLS_MDLS:
    preds = mdl.predict(Xte if use_imp else X_te)
    acc   = accuracy_score(ycls_te, preds)
    cls_res[name] = dict(accuracy=acc, preds=preds)
    print(f"{name:<26} {acc:>10.4f}")

best_cls = max(cls_res, key=lambda k: cls_res[k]["accuracy"])
print(f"\nBest classifier: {best_cls}  Acc={cls_res[best_cls]['accuracy']:.4f}")
print("\nDetailed report:")
print(classification_report(ycls_te, cls_res[best_cls]["preds"],
                             target_names=le3.classes_, zero_division=0))

# ══════════════════════════════════════════════════════════════════════
# 8.  SHAP
# ══════════════════════════════════════════════════════════════════════
print("Computing SHAP values ...")
exp_r  = shap.TreeExplainer(xgb)
sv_r   = exp_r.shap_values(Xte)

exp_c  = shap.TreeExplainer(xgbc)
sv_c   = exp_c.shap_values(Xte)
if sv_c.ndim == 3:
    sv_c_abs = np.abs(sv_c).mean(axis=2)
else:
    sv_c_abs = np.abs(sv_c)
print("SHAP done.")

# ══════════════════════════════════════════════════════════════════════
# 9.  VISUALISATION
# ══════════════════════════════════════════════════════════════════════
print("Building dashboard ...")

C = {"bg0":"#0f1117","bg1":"#1a1d27","ax":"#1e2130",
     "cyan":"#00d4ff","gold":"#ffd700","green":"#00e676",
     "pink":"#ff4081","purple":"#ce93d8","text":"#e8eaf6","muted":"#9e9e9e"}

def ax_style(ax, title="", xlabel="", ylabel=""):
    ax.set_facecolor(C["ax"])
    for sp in ax.spines.values(): sp.set_color("#333")
    ax.tick_params(colors=C["muted"], labelsize=8)
    ax.xaxis.label.set_color(C["muted"]); ax.yaxis.label.set_color(C["muted"])
    if title:  ax.set_title(title,  color=C["text"], fontsize=9, fontweight="bold", pad=7)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)

fig = plt.figure(figsize=(22, 36))
fig.patch.set_facecolor(C["bg0"])
gs  = gridspec.GridSpec(5, 3, figure=fig, hspace=0.50, wspace=0.35)

# Row 0: data overview
ax = fig.add_subplot(gs[0, 0])
ax_style(ax, "Pre-Berth Delay Distribution", "Delay (hours)", "Count")
ax.hist(y_raw_s, bins=50, color=C["cyan"], alpha=0.75, edgecolor="#111")
for p, col in [(25,C["gold"]),(50,C["green"]),(75,C["pink"])]:
    v = np.percentile(y_raw_s, p)
    ax.axvline(v, color=col, lw=1.5, linestyle="--")
    ax.text(v+2, ax.get_ylim()[1]*0.78, f"P{p}={v:.0f}h", color=col, fontsize=7)

ax = fig.add_subplot(gs[0, 1])
ax_style(ax, "log(Delay) — Near-Normal (justifies log-regression)", "log(hours)", "Count")
ax.hist(y_log_s, bins=40, color=C["purple"], alpha=0.8, edgecolor="#111")
mu, sd = y_log_s.mean(), y_log_s.std()
xs = np.linspace(y_log_s.min(), y_log_s.max(), 200)
ax.plot(xs, norm.pdf(xs,mu,sd)*len(y_log_s)*(xs[1]-xs[0]),
        color=C["gold"], lw=2, label=f"N({mu:.2f},{sd:.2f})")
ax.legend(fontsize=8, labelcolor=C["muted"])

ax = fig.add_subplot(gs[0, 2])
ax_style(ax, "Class Distribution (3-class)", "", "")
cnts = [np.sum(y_cls_s==i) for i in range(3)]
cols_cls = [C["green"], C["gold"], C["pink"]]
bars = ax.barh(LABELS_3, cnts, color=cols_cls)
for bar, cnt in zip(bars, cnts):
    ax.text(cnt+2, bar.get_y()+bar.get_height()/2,
            f"{cnt} ({cnt/len(y_cls_s)*100:.0f}%)", va="center", color=C["text"], fontsize=8)

# Row 1: regression results
ax = fig.add_subplot(gs[1, 0])
ax_style(ax, f"Predicted vs Actual — {best_reg}", "Actual (h)", "Predicted (h)")
pr = reg_res[best_reg]["praw"]
ax.scatter(yraw_te, pr, alpha=0.4, s=18, color=C["cyan"], edgecolors="none")
lim = (0, max(yraw_te.max(), pr.max())*1.05)
ax.plot(lim, lim, color=C["gold"], lw=1.5)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.text(0.05,0.90, f"MAE={reg_res[best_reg]['mae']:.1f}h\nR2(log)={reg_res[best_reg]['r2log']:.3f}",
        transform=ax.transAxes, color=C["text"], fontsize=9, va="top",
        bbox=dict(boxstyle="round",fc="#222",ec="#555"))

ax = fig.add_subplot(gs[1, 1])
ax_style(ax, "Regression MAE — All Models (Test Set)", "MAE (hours)", "")
rnames = [n for n,_ in REG_MDLS]
rmaes  = [reg_res[n]["mae"] for n in rnames]
rcols  = [C["green"] if n==best_reg else C["cyan"] for n in rnames]
bars   = ax.barh(rnames, rmaes, color=rcols)
for bar, v in zip(bars, rmaes):
    ax.text(v+0.3, bar.get_y()+bar.get_height()/2, f"{v:.1f}h", va="center", color=C["text"], fontsize=8)

ax = fig.add_subplot(gs[1, 2])
ax_style(ax, "R2 on log(delay) — All Models", "R2 (log scale)", "")
r2ls  = [reg_res[n]["r2log"] for n in rnames]
rc2   = [C["green"] if n==best_reg else C["purple"] for n in rnames]
bars  = ax.barh(rnames, r2ls, color=rc2)
for bar, v in zip(bars, r2ls):
    ax.text(v+0.005, bar.get_y()+bar.get_height()/2, f"{v:.3f}", va="center", color=C["text"], fontsize=8)
ax.set_xlim(0,1)

# Row 2: classification + residuals
ax = fig.add_subplot(gs[2, 0])
ax_style(ax, f"Confusion Matrix — {best_cls}")
cm   = confusion_matrix(ycls_te, cls_res[best_cls]["preds"])
disp = ConfusionMatrixDisplay(cm, display_labels=le3.classes_)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_facecolor(C["ax"])
ax.tick_params(colors=C["muted"], labelsize=7)
ax.set_xlabel("Predicted", color=C["muted"]); ax.set_ylabel("True", color=C["muted"])

ax = fig.add_subplot(gs[2, 1])
ax_style(ax, "Classification Accuracy (Test Set)", "Accuracy", "")
cnames = [n for n,_,_ in CLS_MDLS]
accs   = [cls_res[n]["accuracy"] for n in cnames]
ccols  = [C["green"] if n==best_cls else C["cyan"] for n in cnames]
bars   = ax.barh(cnames, accs, color=ccols)
for bar, v in zip(bars, accs):
    ax.text(v+0.003, bar.get_y()+bar.get_height()/2, f"{v:.3f}", va="center", color=C["text"], fontsize=8)
ax.set_xlim(0,1)
ax.axvline(1/3, color=C["gold"], lw=1, linestyle="--", label="Random (33%)")
ax.legend(fontsize=8, labelcolor=C["muted"])

ax = fig.add_subplot(gs[2, 2])
ax_style(ax, "Residual Distribution — Best Regressor", "Residual (h)", "Count")
resid = reg_res[best_reg]["praw"] - yraw_te
ax.hist(resid, bins=35, color=C["green"], alpha=0.8, edgecolor="#111")
ax.axvline(0, color=C["gold"], lw=1.5, linestyle="--")
ax.axvline(np.mean(resid), color=C["pink"], lw=1.5, linestyle="-.",
           label=f"Bias={np.mean(resid):.1f}h")
ax.legend(fontsize=8, labelcolor=C["muted"])

# Row 3: SHAP
ax = fig.add_subplot(gs[3, :2])
ax_style(ax, "SHAP Feature Importance — XGBoost Regression (mean |SHAP| on log-delay scale)", "", "")
si_r = pd.Series(np.abs(sv_r).mean(0), index=FEAT_NAMES).sort_values(ascending=False).head(18)
rev  = si_r.iloc[::-1]
ax.barh(rev.index, rev.values, color=C["gold"])
ax.set_xlabel("Mean |SHAP value|", color=C["muted"])

ax = fig.add_subplot(gs[3, 2])
ax_style(ax, "SHAP — XGBoost Classifier (mean across classes)", "Mean |SHAP value|", "")
si_c  = pd.Series(sv_c_abs.mean(0), index=FEAT_NAMES).sort_values(ascending=False).head(15)
rev2  = si_c.iloc[::-1]
ax.barh(rev2.index, rev2.values, color=C["cyan"])

# Row 4: error analysis
ax = fig.add_subplot(gs[4, 0])
ax_style(ax, "Absolute Error vs Actual Delay", "Actual Delay (h)", "Abs Error (h)")
ax.scatter(yraw_te, np.abs(resid), alpha=0.35, s=14, color=C["purple"], edgecolors="none")
bins_e = np.percentile(yraw_te, np.linspace(0, 100, 11))
midpt  = [(bins_e[i]+bins_e[i+1])/2 for i in range(len(bins_e)-1)]
med_e  = [np.median(np.abs(resid)[(yraw_te>=bins_e[i])&(yraw_te<bins_e[i+1])])
          for i in range(len(bins_e)-1)]
ax.plot(midpt, med_e, color=C["gold"], lw=2, marker="o", ms=5, label="Median error")
ax.legend(fontsize=8, labelcolor=C["muted"])

ax = fig.add_subplot(gs[4, 1])
ax_style(ax, "Pre-Berth Delay by Vessel Category", "Category", "Delay (h)")
cat_order  = ["Tanker","DryBulk","Container","BreakBulk","Unknown"]
cat_delays = [y_raw_s[df_s["vessel_cat"].values == c] for c in cat_order]
bp = ax.boxplot(cat_delays, labels=cat_order, patch_artist=True,
                medianprops=dict(color=C["gold"],lw=2),
                whiskerprops=dict(color=C["muted"]),
                capprops=dict(color=C["muted"]),
                flierprops=dict(markerfacecolor=C["pink"],ms=3,alpha=0.5))
for patch, col in zip(bp["boxes"], [C["cyan"],C["green"],C["purple"],C["gold"],C["muted"]]):
    patch.set_facecolor(col); patch.set_alpha(0.6)
ax.tick_params(axis="x", labelsize=7)

ax = fig.add_subplot(gs[4, 2])
ax_style(ax, "Tide Coefficient vs Pre-Berth Delay", "Tide Coefficient", "log(Delay h)")
tc   = df_s["arrival_tide_coefficient"].values
mask = ~np.isnan(tc)
sc   = ax.scatter(tc[mask], np.log(y_raw_s[mask]), alpha=0.35, s=12,
                  c=np.log(y_raw_s[mask]), cmap="plasma", edgecolors="none")
z    = np.polyfit(tc[mask], np.log(y_raw_s[mask]), 1)
xs   = np.linspace(tc[mask].min(), tc[mask].max(), 200)
ax.plot(xs, np.poly1d(z)(xs), color=C["gold"], lw=2, label=f"slope={z[0]:.3f}")
ax.legend(fontsize=8, labelcolor=C["muted"])
plt.colorbar(sc, ax=ax).ax.tick_params(colors=C["muted"])

fig.suptitle("Pre-Berth Delay Prediction  |  Haldia Dock Complex\n"
             "Log-regression + 3-class classification  |  Chronological 80/20 split",
             color=C["text"], fontsize=14, fontweight="bold", y=0.999)

plt.savefig("../outputs/pre_berth_delay_analysis.png",
            dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
print("Dashboard saved.")

# ══════════════════════════════════════════════════════════════════════
# 10.  FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("FINAL RESULTS — TEST SET")
print("=" * 70)

print("\nREGRESSION (log-transform target; metrics on raw hours):")
print(f"  {'Model':<26} {'MAE(h)':>8} {'RMSE(h)':>9} {'R2(raw)':>8} {'R2(log)':>8}")
print("  " + "-" * 58)
for n,_ in REG_MDLS:
    r = reg_res[n]; s = " *" if n==best_reg else ""
    print(f"  {n:<26} {r['mae']:>8.2f} {r['rmse']:>9.2f} {r['r2']:>8.4f} {r['r2log']:>8.4f}{s}")

print("\nCLASSIFICATION (3 delay bins):")
print(f"  {'Model':<26} {'Accuracy':>10}")
print("  " + "-" * 38)
for n,_,_ in CLS_MDLS:
    s = " *" if n==best_cls else ""
    print(f"  {n:<26} {cls_res[n]['accuracy']:>10.4f}{s}")

print("\nDelay bins used:")
print("  Short  : <= 24h    (same-day or next-tide entry)")
print("  Medium : 24-120h   (within tidal-fortnight window)")
print("  Long   : > 120h    (neap-trap / congestion / doc delays)")

print("\nDone.")

STEP 1 — LOAD & FILTER
Usable rows: 1,343  (from 1,682 raw)

STEP 2 — FEATURE ENGINEERING (literature-backed)
Features engineered.
Feature matrix: 1343 rows x 36 features

Target distribution:
count    1343.00
mean       69.14
std        63.77
min         4.50
25%        23.59
50%        45.50
75%        92.55
max       333.20
Name: pre_berth_delay_hr, dtype: float64

Class counts:
  Short (<=1d)        :  241  (17.9%)
  Medium (1-5d)       :  758  (56.4%)
  Long (>5d)          :  344  (25.6%)

STEP 3 — CHRONOLOGICAL SPLIT (train=80%, test=20%)
Train: 1,074  |  Test: 269

STEP 4 — REGRESSION on log(pre_berth_delay_hr)

Model                        MAE(h)   RMSE(h)  R2(raw)  R2(log)
----------------------------------------------------------------
Ridge (baseline)              41.70     59.48  -0.0181   0.0720
Random Forest                 37.91     54.31   0.1513   0.2044
XGBoost                       38.71     55.27   0.1208   0.2020
LightGBM                      36.97     52.35   0.21